# 유튜브 인기동영상 데이터

In [ ]:
import pandas as pd
# 1. 인기동영상 제작횟수가 많은 채널 상위 10개명을 출력하라 (날짜기준, 중복포함)
dataurl = 'https://raw.githubusercontent.com/Datamanim/datarepo/main/youtube/youtube.csv'
df = pd.read_csv(dataurl, index_col=0)
answer = df.loc[df['channelId'].isin(df['channelId'].value_counts().head(10).index)]['channelTitle'].unique()

In [ ]:
# 2. 논란으로 인기동영상이 된 케이스를 확인하고 싶다. dislikes수가 like 수보다 높은 동영상을 제작한 채널을 모두 출력하라
answer = df.loc[df['dislikes'] > df['likes']]['channelTitle'].unique()

In [ ]:
# 3. 채널명을 바꾼 케이스가 있는지 확인하고 싶다. channelId의 경우 고유값이므로 이를 통해 채널명을 한번이라도 바꾼 채널의 갯수를 구하여라
change = df[['channelTitle','channelId']].drop_duplicates()['channelId'].value_counts()
target = change.loc[change > 1]
answer = len(target)

In [ ]:
# 4. 일요일에 인기있었던 영상들중 가장많은 영상 종류(categoryId)는 무엇인가?
df['trending_date2'] = pd.to_datetime(df['trending_date2'])
answer = df.loc[df['trending_date2'].dt.day_name() == 'Sunday']['categoryId'].value_counts().index[0]
answer = df.loc[df['trending_date2'].astype('datetime64[ns]').dt.day_of_week == 6]['categoryId'].value_counts().index[0]

In [ ]:
# 5. 각 요일별 인기 영상들의 categoryId는 각각 몇개 씩인지 하나의 데이터 프레임으로 표현하라
group = df.groupby([df['trending_date2'].dt.day_name(), 'categoryId'], as_index=False).size()
answer = group.pivot(index='categoryId', columns='trending_date2')

In [ ]:
# 6. 댓글의 수로 (comment_count) 영상 반응에 대한 판단을 할 수 있다. viewcount대비 댓글수가 가장 높은 영상을 확인하라 (view_count값이 0인 경우는 제외한다)
target2 = df.loc[df['view_count'] != 0]
t = target2.copy()
t['ratio'] = (target2['comment_count'] / target2['view_count']).dropna()
answer = t.sort_values(by='ratio',  ascending=False).iloc[0]['title']

In [ ]:
# 7. 댓글의 수로 (comment_count) 영상 반응에 대한 판단을 할 수 있다.viewcount대비 댓글수가 가장 낮은 영상을 확인하라 (view_counts, ratio값이 0인경우는 제외한다.)
target2 = df.loc[df['view_count'] != 0]
t = target2.copy()
t['ratio'] = (target2['comment_count'] / target2['view_count']).dropna()
answer = t.loc[t['ratio'] > 0].sort_values(by='ratio', ascending=True).iloc[0]['title']

ratio = (df['comment_count'] / df['view_count']).dropna().sort_values()
answer = df.loc[ratio[ratio != 0].index[0]]['title']

In [ ]:
# 8. like 대비 dislike의 수가 가장 적은 영상은 무엇인가? (like, dislike 값이 0인경우는 제외한다)
target = df.loc[(df['dislikes'] > 0) & (df['likes'] > 0)]
num = (target['dislikes'] / target['likes']).dropna().sort_values().index[0]
answer = df.loc[num]['title']

In [ ]:
# 9. 가장많은 트렌드 영상을 제작한 채널의 이름은 무엇인가? (날짜기준, 중복포함)
answer = df.loc[df['channelId'] == df['channelId'].value_counts().index[0]]['channelTitle'].unique()[0]

In [ ]:
# 10. 20회(20일)이상 인기동영상 리스트에 포함된 동영상의 숫자는?
answer = (df[['title','channelId']].value_counts() >= 20).sum()

# 유튜브 공범컨텐츠 동영상 데이터

In [ ]:
import pandas as pd
# 비디오 정보
dataurl1 = 'https://raw.githubusercontent.com/Datamanim/datarepo/main/youtube/channelInfo.csv'
# 참가자 채널 정보
dataurl2 = 'https://raw.githubusercontent.com/Datamanim/datarepo/main/youtube/videoInfo.csv'
channel = pd.read_csv(dataurl1)
video = pd.read_csv(dataurl2)

In [ ]:
# 11. 각 데이터의 ‘ct’컬럼을 시간으로 인식할수 있게 datatype을 변경하고 video 데이터의 videoname의 각 value 마다 몇개의 데이터씩 가지고 있는지 확인하라
channel['ct'] = pd.to_datetime(channel['ct'])
video['ct'] = pd.to_datetime(video['ct'])
answer = video['videoname'].value_counts()

In [ ]:
# 12. 수집된 각 video의 가장 최신화 된 날짜의 viewcount값을 출력하라
answer = video.sort_values(by=['videoname','ct']).drop_duplicates('videoname', keep='last')[['viewcnt','videoname','ct']].reset_index(drop=True)

In [ ]:
# 13. Channel 데이터중 2021-10-03일 이후 각 채널의 처음 기록 됐던 구독자 수(subcnt)를 출력하라
target = channel.loc[channel['ct'] >= '2021-10-03'].sort_values(by=['ct','channelname']).drop_duplicates('channelname')
answer = target[['channelname','subcnt']].reset_index(drop=True)

In [ ]:
# 14. 각채널의 2021-10-03 03:00:00 ~ 2021-11-01 15:00:00 까지 구독자수 (subcnt) 의 증가량을 구하여라
start = channel.loc[channel['ct'].dt.strftime('%Y-%m-%d %H') == '2021-10-03 03']
end = channel.loc[channel['ct'].dt.strftime('%Y-%m-%d %H') == '2021-11-01 15']

start_df = start[['channelname','subcnt']].reset_index(drop=True)
end_df = end[['channelname','subcnt']].reset_index(drop=True)

start_df.columns = ['channelname','start_sub']
end_df.columns = ['channelname','end_sub']

tt = pd.merge(start_df, end_df)
tt['del'] = tt['end_sub'] - tt['start_sub']
answer = tt[['channelname','del']]

In [ ]:
target = channel.loc[(channel['ct']>='2021-10-03 03') & (channel['ct']<='2021-11-01 15')]

start_df = target[['channelname','subcnt']].drop_duplicates('channelname', keep='first').reset_index(drop=True)
end_df = target[['channelname','subcnt']].drop_duplicates('channelname', keep='last').reset_index(drop=True)

start_df.columns = ['channelname','start_sub']
end_df.columns = ['channelname','end_sub']

tt = pd.merge(start_df, end_df)
tt['del'] = tt['end_sub'] - tt['start_sub']
answer = tt[['channelname','del']]

In [ ]:
# 15. 각 비디오는 10분 간격으로 구독자수, 좋아요, 싫어요수, 댓글수가 수집된것으로 알려졌다. 
# 공범 EP1의 비디오정보 데이터중 수집간격이 5분 이하, 20분이상인 데이터 구간( 해당 시점 전,후) 의 시각을 모두 출력하라
import datetime
ep_one = video.loc[video.videoname.str.contains('1')].sort_values('ct').reset_index(drop=True)
ep_one[(ep_one.ct.diff(1) >=datetime.timedelta(minutes=20)) | (ep_one.ct.diff(1) <=datetime.timedelta(minutes=5))]
answer = ep_one[ep_one.index.isin([720,721,722,723,1635,1636,1637])]

ep_one = video.loc[video['videoname'] == ' 공범 EP1'].sort_values('ct').reset_index(drop=True)
ep_one_index = ep_one.loc[(ep_one['ct'].diff(1) <= '00:05:00') | (ep_one['ct'].diff(1) >= '00:20:00')].index
answer = ep_one.iloc[ep_one.index.isin(set(ep_one_index-1) | set(ep_one_index) | set(ep_one_index+1))]

In [ ]:
# 16. 각 에피소드의 시작날짜(년-월-일)를 에피소드 이름과 묶어 데이터 프레임으로 만들고 출력하라
start_date = video.sort_values(['ct','videoname']).drop_duplicates('videoname')[['ct','videoname']]
start_date['date'] = start_date['ct'].dt.date
answer = start_date[['date','videoname']]

In [ ]:
# 17. “공범” 컨텐츠의 경우 19:00시에 공개 되는것으로 알려져있다.
# 공개된 날의 21시의 viewcnt, ct, videoname 으로 구성된 데이터 프레임을 viewcnt를 내림차순으로 정렬하여 출력하라
video['time'] = video['ct'].dt.hour
answer = video.loc[video['time'] == 21].sort_values(['videoname','ct']).drop_duplicates('videoname')\
    .sort_values('viewcnt', ascending=False)[['videoname','viewcnt','ct']]\
    .reset_index(drop=True)

answer = video.loc[video['ct'].dt.hour == 21].sort_values(['videoname','ct'])[['videoname','viewcnt','ct']]\
    .drop_duplicates('videoname').sort_values('viewcnt', ascending=False).reset_index(drop=True)

In [ ]:
# 18. video 정보의 가장 최근 데이터들에서 각 에피소드의 싫어요/좋아요 비율을 ratio 컬럼으로 만들고 
# videoname, ratio로 구성된 데이터 프레임을 ratio를 오름차순으로 정렬하라
target = video.sort_values('ct').drop_duplicates('videoname', keep='last')
target['ratio'] = target['dislikecnt'] / target['likecnt']
answer = target.sort_values('ratio')[['videoname','ratio']].reset_index(drop=True)

In [ ]:
# 19. 2021-11-01 00:00:00 ~ 15:00:00까지 각 에피소드별 viewcnt의 증가량을 데이터 프레임으로 만드시오
target = video.loc[(video['ct'] >= '2021-11-01') & (video['ct'] <= '2021-11-01 15')].reset_index(drop=True)

def check(x):
    result = max(x) - min(x)
    return result

answer = target[['videoname','viewcnt']].groupby('videoname').agg(check)

In [ ]:
# 20. video 데이터 중에서 중복되는 데이터가 존재한다. 중복되는 각 데이터의 시간대와 videoname 을 구하여라
answer = video.loc[video.index.isin(set(video.index) - set(video.drop_duplicates().index))][['videoname','ct']]
answer = video[video.duplicated(subset=['ct','videoname'])][['videoname','ct']]

# 월드컵 출전선수 골기록 데이터

In [137]:
import pandas as pd
# 1930 ~ 2018년도 월드컵 출전선수 골기록
dataurl = 'https://raw.githubusercontent.com/Datamanim/datarepo/main/worldcup/worldcupgoals.csv'
df = pd.read_csv(dataurl)

In [2]:
# 21. 주어진 전체 기간의 각 나라별 골득점수 상위 5개 국가와 그 득점수를 데이터프레임형태로 출력하라
answer = df.groupby('Country')[['Goals']].sum().sort_values('Goals', ascending=False).head(5)

In [18]:
# 22. 주어진 전체기간동안 골득점을 한 선수가 가장 많은 나라 상위 5개 국가와 그 선수 숫자를 데이터 프레임 형식으로 출력하라
answer = df.groupby('Country').size().sort_values(ascending=False).head(5)

In [31]:
# 23. Years 컬럼은 년도 -년도 형식으로 구성되어있고, 각 년도는 4자리 숫자이다. 년도 표기가 4자리 숫자로 안된 케이스가 존재한다. 해당 건은 몇건인지 출력하라
df['yearLst'] = df['Years'].str.split('-')

def checkFour(x):
    for value in x:
        if len(str(value)) != 4:
            return False
    return True

df['check'] = df['yearLst'].apply(checkFour)
answer = len(df[~df['check']])

In [44]:
# 24. Q3에서 발생한 예외 케이스를 제외한 데이터프레임을 df2라고 정의하고 데이터의 행의 숫자를 출력하라 (아래 문제부터는 df2로 풀이하겠습니다)
df2 = df.loc[df['check']].reset_index(drop=True)

In [52]:
# 25. 월드컵 출전횟수를 나타내는 ‘LenCup’ 컬럼을 추가하고 4회 출전한 선수의 숫자를 구하여라
df2['LenCup'] = df2['yearLst'].str.len()
answer = df2['LenCup'].value_counts()[4]

In [84]:
# 26. Yugoslavia 국가의 월드컵 출전횟수가 2회인 선수들의 숫자를 구하여라
answer = len(df2.loc[(df2['Country'] == 'Yugoslavia') & (df2['LenCup'] == 2)])

In [85]:
# 27. 2002년도에 출전한 전체 선수는 몇명인가?
answer = len(df2.loc[df2['Years'].str.contains('2002')])

In [ ]:
# 28. 이름에 ‘carlos’ 단어가 들어가는 선수의 숫자는 몇 명인가? (대, 소문자 구분 x)
answer = len(df2.loc[df2['Player'].str.lower().str.contains('carlos')])

In [119]:
# 29. 월드컵 출전 횟수가 1회뿐인 선수들 중에서 가장 많은 득점을 올렸던 선수는 누구인가?
answer = df2.loc[df2['LenCup'] == 1].sort_values(by='Goals', ascending=False)['Player'].values[0]

In [136]:
# 30. 월드컵 출전횟수가 1회 뿐인 선수들이 가장 많은 국가는 어디인가?
answer = df2.loc[df2['LenCup'] == 1]['Country'].value_counts().index[0]

# 서울시 따릉이 이용정보 데이터

In [1]:
import pandas as pd
# 서울특별시_공공자전거 시간대별 이용정보
df = pd.read_csv('https://raw.githubusercontent.com/Datamanim/datarepo/main/bicycle/seoul_bi.csv')

In [2]:
# 31. 대여일자별 데이터의 수를 데이터프레임으로 출력하고, 가장 많은 데이터가 있는 날짜를 출력하라
result = df['대여일자'].value_counts().sort_index().to_frame()
answer = result.loc[result['count'] == result['count'].max()].index[0]

In [3]:
# 32. 각 일자의 요일을 표기하고 (‘Monday’ ~’Sunday’) ‘day_name’컬럼을 추가하고 이를 이용하여 각 요일별 이용 횟수의 총합을 데이터 프레임으로 출력하라
df['대여일자'] = pd.to_datetime(df['대여일자'])
df['day_name'] = df['대여일자'].dt.day_name()
answer = df['day_name'].value_counts().to_frame()

In [4]:
# 33. 각 요일별 가장 많이 이용한 대여소의 이용횟수와 대여소 번호를 데이터 프레임으로 출력하라
result = df.groupby(['day_name','대여소번호']).size().to_frame('size').sort_values(by=['day_name','size'], ascending=False).reset_index()
answer = result.drop_duplicates('day_name', keep='first').reset_index(drop=True)

In [42]:
# 34. 나이대별 대여구분 코드의 (일일권/전체횟수) 비율을 구한 후 가장 높은 비율을 가지는 나이대를 확인하라. 
# 일일권의 경우 일일권 과 일일권(비회원)을 모두 포함하라
daily = df.loc[df['대여구분코드'].isin(['일일권','일일권(비회원)'])]['연령대코드'].value_counts().sort_index()
total = df['연령대코드'].value_counts().sort_index()
answer = (daily / total).sort_values(ascending=False).index[0]

In [62]:
# 35. 연령대별 평균 이동거리를 구하여라
answer = df.groupby('연령대코드')[['이동거리']].mean()

In [109]:
# 36. 연령대 코드가 20대인 데이터를 추출하고,이동거리값이 추출한 데이터의 이동거리값의 평균 이상인 데이터를 추출한다.
# 최종 추출된 데이터를 대여일자, 대여소 번호 순서로 내림차순 정렬 후 1행부터 200행까지의 탄소량의 평균을 소숫점 3째 자리까지 구하여라
tw = df.loc[df['연령대코드']=='20대'].reset_index(drop=True)
tw_mean = tw.loc[tw['이동거리'] >= tw['이동거리'].mean()].reset_index(drop=True)
tw_mean['탄소량'] = tw_mean['탄소량'].astype('float')
target = tw_mean.sort_values(by=['대여일자','대여소번호'], ascending=False).reset_index(drop=True).iloc[:200]['탄소량']
answer = round(target.mean(), 3)

In [145]:
# 37. 6월 7일 ~10대의 “이용건수”의 중앙값은?
answer = df.loc[(df['대여일자']=='2021-06-07')&(df['연령대코드']=='~10대')]['이용건수'].sort_values().median()

In [232]:
# 38. 평일 (월~금) 출근 시간대(오전 6,7,8시)의 대여소별 이용 횟수를 구해서 데이터 프레임 형태로 표현한 후 
# 각 대여시간별 이용 횟수의 상위 3개 대여소와 이용횟수를 출력하라
target = df[~(df['대여일자'].dt.day_of_week.isin([5,6]))&(df['대여시간'].isin([6,7,8]))].reset_index(drop=True)
result = target.groupby(['대여시간','대여소번호'])['이용건수'].size().to_frame('이용횟수')
answer = result.sort_values(by=['대여시간','이용횟수'], ascending=False).groupby('대여시간').head(3)

In [240]:
# 39. 이동거리의 평균 이상의 이동거리 값을 가지는 데이터를 추출하여 추출데이터의 이동거리의 표본표준편차 값을 구하여라
answer = df.loc[df['이동거리'] >= df['이동거리'].mean()]['이동거리'].std()

In [262]:
# 40. 남성(‘M’ or ‘m’)과 여성(‘F’ or ‘f’)의 이동거리값의 평균값을 구하여라
df['sex'] = df['성별'].map(lambda x: '남' if x in ['M','m'] else '여')
answer = df[['sex','이동거리']].groupby('sex').mean()
# answer = df.groupby(df['성별'].str.lower())['이동거리'].mean().to_frame()

# 전세계 행복도 지표

In [264]:
import pandas as pd
# 전세계 행복도 지표 조사
df = pd.read_csv('https://raw.githubusercontent.com/Datamanim/datarepo/main/happy2/happiness.csv', encoding='utf-8')

In [293]:
# 41. 데이터는 2018년도와 2019년도의 전세계 행복 지수를 표현한다. 각년도의 행복랭킹 10위를 차지한 나라의 행복점수의 평균을 구하여라
answer = df.loc[df['행복랭킹']==10][['점수']].mean()

In [298]:
# 42. 데이터는 2018년도와 2019년도의 전세계 행복 지수를 표현한다. 각년도의 행복랭킹 50위이내의 나라들의 각각의 행복점수 평균을 데이터프레임으로 표시하라
answer = df.loc[df['행복랭킹'] <= 50].groupby('년도')['점수'].mean()

In [307]:
df

,행복랭킹,나라명,점수,상대GDP,사회적지원,행복기대치,선택의 자유도,관대함,부패에 대한인식,년도
0,1,Finland,7.769,1.340,1.587,0.986,0.596,0.153,0.393,2019
1,2,Denmark,7.600,1.383,1.573,0.996,0.592,0.252,0.410,2019
2,3,Norway,7.554,1.488,1.582,1.028,0.603,0.271,0.341,2019
3,4,Iceland,7.494,1.380,1.624,1.026,0.591,0.354,0.118,2019
4,5,Netherlands,7.488,1.396,1.522,0.999,0.557,0.322,0.298,2019
...,...,...,...,...,...,...,...,...,...,...
307,152,Yemen,3.355,0.442,1.073,0.343,0.244,0.083,0.064,2018
308,153,Tanzania,3.303,0.455,0.991,0.381,0.481,0.270,0.097,2018
309,154,South Sudan,3.254,0.337,0.608,0.177,0.112,0.224,0.106,2018
310,155,Central African Republic,3.083,0.024,0.000,0.010,0.305,0.218,0.038,2018


In [317]:
# 43. 2018년도 데이터들만 추출하여 행복점수와 부패에 대한 인식에 대한 상관계수를 구하여라
answer = df.loc[df['년도']==2018][['점수','부패에 대한인식']].corr().iloc[0,1]

In [318]:
# 44. 2018년도와 2019년도의 행복랭킹이 변화하지 않은 나라명의 수를 구하여라